# StreamGuard Stage 5 + Stage 6: Node Embedding & Graph Tensors (GPU)

**Purpose**: Run Stage 5 (CodeBERT node embedding) and Stage 6 (graph tensor assembly) on Colab GPU.

**Why Colab?** Stage 5 runs CodeBERT inference on ~1.4M node statements. On CPU this takes ~87 hours. On T4 GPU: **~2-4 hours**.

**Prerequisites**:
1. CPG JSON files zipped and uploaded to Google Drive
2. This notebook uploaded to Google Colab
3. Runtime set to **GPU** (T4 or better)

---

## Step-by-Step Process

| Step | What | Time |
|------|------|------|
| 1 | Mount Drive + unzip CPG data | 2-5 min |
| 2 | Clone repo + install deps | 2-3 min |
| 3 | Run Stage 5 (embedding) | 2-4 hours (T4) |
| 4 | Run Stage 6 (graph tensors) | 5-10 min |
| 5 | Zip outputs + copy to Drive | 5-10 min |

## Cell 1: Verify GPU Runtime

In [ ]:
import torch
import subprocess

if not torch.cuda.is_available():
    raise RuntimeError(
        "NO GPU DETECTED!\n"
        "Go to: Runtime > Change runtime type > Hardware accelerator > GPU (T4)\n"
        "Then restart the runtime and re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# Quick CUDA test
x = torch.randn(100, 768, device='cuda')
print(f"CUDA tensor test: OK ({x.shape})")

## Cell 2: Mount Google Drive

Your Drive should contain the CPG zip at:
```
My Drive/StreamGuard/cpg_data.zip
```

**How to prepare this zip locally** (run on your Windows machine before starting Colab):
```bash
cd C:\Users\Vimal Sajan\streamguard
python -c "import shutil; shutil.make_archive('cpg_data', 'zip', '.', 'training/data/processed/cpg')"
```
Then upload `cpg_data.zip` to `My Drive/StreamGuard/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Configure paths ──────────────────────────────────────────────
# Change this if your zip is in a different Drive folder
DRIVE_FOLDER = '/content/drive/MyDrive/StreamGuard'
CPG_ZIP = f'{DRIVE_FOLDER}/cpg_data.zip'

# Local working directories (on Colab's fast SSD)
WORK_DIR = '/content/streamguard'
CPG_DIR = f'{WORK_DIR}/training/data/processed/cpg'
EMBED_DIR = f'{WORK_DIR}/training/data/processed/embedded'
GRAPH_DIR = f'{WORK_DIR}/training/data/graphs'

# Verify zip exists
if not os.path.exists(CPG_ZIP):
    raise FileNotFoundError(
        f"CPG zip not found at: {CPG_ZIP}\n"
        f"Upload cpg_data.zip to Google Drive > StreamGuard folder first.\n"
        f"See instructions above this cell."
    )

zip_size_mb = os.path.getsize(CPG_ZIP) / 1024 / 1024
print(f"Found CPG zip: {zip_size_mb:.0f} MB")

## Cell 3: Clone Repo + Install Dependencies

In [ ]:
%%bash
# Clone repo (if not already cloned)
if [ ! -d "/content/streamguard" ]; then
    git clone https://github.com/VimalSajanGeorge/streamguard.git /content/streamguard
else
    cd /content/streamguard && git pull
fi

In [ ]:
# Install only what Stage 5 + 6 need (minimal, fast)
!pip install -q transformers==4.44.0 loguru h5py numpy 'torch>=2.2.0'

# Verify imports
import transformers, loguru, h5py, numpy, torch
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"h5py: {h5py.__version__}")
print(f"numpy: {numpy.__version__}")

## Cell 4: Unzip CPG Data to Local SSD

This extracts to Colab's local disk (`/content/`) which is much faster than reading from Drive.

In [ ]:
import zipfile
import time

os.makedirs(CPG_DIR, exist_ok=True)

# Check if already extracted
existing = sum(1 for _ in os.scandir(CPG_DIR) if _.is_dir())
if existing > 200:  # 256 shard dirs expected
    print(f"CPG data already extracted ({existing} shard dirs). Skipping.")
else:
    print(f"Extracting {CPG_ZIP} ...")
    start = time.time()
    with zipfile.ZipFile(CPG_ZIP, 'r') as z:
        z.extractall(f'{WORK_DIR}/training/data/processed/')
    elapsed = time.time() - start
    print(f"Extracted in {elapsed:.0f}s")

    # Phase 3: Also check for with_cfa/ CPG zip (CFA-augmented samples)
    CFA_CPG_ZIP = f'{DRIVE_FOLDER}/cpg_data_cfa.zip'
    if os.path.exists(CFA_CPG_ZIP):
        print(f"Found CFA CPG zip: {os.path.getsize(CFA_CPG_ZIP)/1024/1024:.0f} MB")
        print(f"Extracting CFA CPGs...")
        start2 = time.time()
        with zipfile.ZipFile(CFA_CPG_ZIP, 'r') as z:
            z.extractall(f'{WORK_DIR}/training/data/processed/')
        print(f"CFA CPGs extracted in {time.time()-start2:.0f}s")
    else:
        print(f"No CFA CPG zip found at {CFA_CPG_ZIP} (optional — skip if Phase 3 CFA not ready)")

# Count CPG files
from pathlib import Path
cpg_files = list(Path(CPG_DIR).rglob('*.json'))
cpg_files = [f for f in cpg_files if f.name not in ('cpg_stats.json', 'cpg_failures.jsonl', 'checkpoint.json')]
print(f"CPG files ready: {len(cpg_files)}")

# Phase 3 note: target 50K-80K CPG files (originals + CFA augmented)
if len(cpg_files) < 30000:
    print(f"WARNING: Only {len(cpg_files)} CPGs found — expected 50K-80K for Phase 3 full run")

## Cell 5: Run Stage 5 — Node Embedding (GPU)

This is the main compute step. On T4 GPU, expect ~2-4 hours for ~35K samples.

**Progress** is printed every 50 samples. If the runtime disconnects, re-run this cell — it automatically skips already-embedded samples.

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

from training.scripts.preprocessing.stage5_embed import run_stage5, verify_embeddings

# ── DRY RUN FIRST (verify setup) ────────────────────────────────
print("=" * 60)
print("DRY RUN — verifying setup")
print("=" * 60)

dry_stats = run_stage5(
    input_dir=CPG_DIR,
    output_dir=EMBED_DIR,
    dry_run=True,
    max_samples=5,
)
print(f"\nDry run OK. {dry_stats.get('would_process', 0)} samples to process.")
print(f"Total CPGs: {dry_stats.get('total_cpgs', 0)}")

In [ ]:
# ── FULL RUN ON GPU ─────────────────────────────────────────────
# Phase 3: processes 50K-80K CPGs (originals + CFA augmented)
# Phase 1 benchmark: 34,691 samples in ~27 min on T4
# Phase 3 estimate: 50K-80K samples in ~40-70 min on T4
# Checkpoint/resume: automatically skips already-embedded .npz files
print("=" * 60)
print("STARTING FULL STAGE 5 RUN (GPU)")
print("=" * 60)

stats = run_stage5(
    input_dir=CPG_DIR,
    output_dir=EMBED_DIR,
    device='cuda',
    batch_size=64,
)

print("\n" + "=" * 60)
print("STAGE 5 COMPLETE")
print("=" * 60)
for k, v in stats.items():
    print(f"  {k}: {v}")

# Phase 3: Show CFA vs original breakdown
if 'cfa_samples' in stats:
    print(f"\n  Source breakdown: {stats['original_samples']} original + {stats['cfa_samples']} CFA")

# Phase 3: Show per-CWE stats if available
import json
cwe_stats_path = f'{EMBED_DIR}/embed_stats_by_cwe.json'
if os.path.exists(cwe_stats_path):
    cwe_stats = json.loads(open(cwe_stats_path).read())
    print(f"\n  Per-CWE stats ({len(cwe_stats)} CWEs):")
    for cwe, info in sorted(cwe_stats.items(), key=lambda x: -x[1]['sample_count']):
        print(f"    {cwe}: n={info['sample_count']} avg_nodes={info['avg_node_count']} taint%={info['avg_taint_node_pct']}")

In [ ]:
# ── VERIFY EMBEDDINGS ───────────────────────────────────────────
verify_embeddings(EMBED_DIR, max_check=5)

## Cell 6: Run Stage 6 — Graph Tensor Assembly

This is fast (~42 graphs/s, CPU-only). Joins CPG edges + embeddings into HDF5.

In [ ]:
from training.scripts.preprocessing.stage6_graphs import run_stage6, verify_h5, check_pair_integrity

os.makedirs(GRAPH_DIR, exist_ok=True)
H5_PATH = f'{GRAPH_DIR}/all_graphs.h5'

# Phase 3: run with --check-pairs to validate CFA pair linkage
stats6 = run_stage6(
    cpg_dir=CPG_DIR,
    embed_dir=EMBED_DIR,
    output_path=H5_PATH,
    check_pairs=True,
)

print("\n" + "=" * 60)
print("STAGE 6 COMPLETE")
print("=" * 60)
for k, v in stats6.items():
    if k == "cwe_distribution":
        print(f"  {k}:")
        for cwe, count in sorted(v.items(), key=lambda x: -x[1]):
            print(f"    {cwe}: {count}")
    elif k == "pair_integrity":
        print(f"  {k}: {v['valid_pairs']}/{v['total_pairs']} valid, {v['broken_pairs']} broken")
    elif k == "reject_reasons":
        if v:
            print(f"  {k}: {v}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# ── VERIFY HDF5 ─────────────────────────────────────────────────
verify_h5(H5_PATH, max_check=5)

## Cell 7: Copy Outputs Back to Google Drive

Zip the embeddings and HDF5, copy to Drive so you can download to your local machine.

In [ ]:
import shutil

os.makedirs(DRIVE_FOLDER, exist_ok=True)

# ── Zip embeddings ────────────────────────────────────────────
print("Zipping embeddings...")
embed_zip = f'{DRIVE_FOLDER}/embedded_data'
shutil.make_archive(embed_zip, 'zip', f'{WORK_DIR}/training/data/processed', 'embedded')
embed_zip_size = os.path.getsize(f'{embed_zip}.zip') / 1024 / 1024
print(f"  embedded_data.zip: {embed_zip_size:.0f} MB")

# ── Copy HDF5 ────────────────────────────────────────────────
print("Copying HDF5...")
shutil.copy2(H5_PATH, f'{DRIVE_FOLDER}/all_graphs.h5')
h5_size = os.path.getsize(H5_PATH) / 1024 / 1024
print(f"  all_graphs.h5: {h5_size:.0f} MB")

# ── Copy stats and logs ──────────────────────────────────────
for stats_name in ['embed_stats.json', 'embed_stats_by_cwe.json']:
    stats_src = f'{WORK_DIR}/training/data/processed/embedded/{stats_name}'
    if os.path.exists(stats_src):
        shutil.copy2(stats_src, f'{DRIVE_FOLDER}/{stats_name}')
        print(f"  Copied {stats_name}")

for graph_file in ['graph_stats.json', 'graph_rejected.log']:
    src = f'{WORK_DIR}/training/data/graphs/{graph_file}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_FOLDER}/{graph_file}')
        print(f"  Copied {graph_file}")

print(f"\nAll outputs saved to Google Drive: {DRIVE_FOLDER}/")
print("Files:")
for f in os.listdir(DRIVE_FOLDER):
    fpath = os.path.join(DRIVE_FOLDER, f)
    if os.path.isfile(fpath):
        print(f"  {f}: {os.path.getsize(fpath)/1024/1024:.1f} MB")

## Cell 8: Summary & Next Steps

After this notebook completes:

1. **Download from Drive** to your local machine:
   - `embedded_data.zip` → extract to `training/data/processed/embedded/`
   - `all_graphs.h5` → copy to `training/data/graphs/all_graphs.h5`

2. **Verify locally**:
   ```bash
   python -c "from training.scripts.preprocessing.stage5_embed import verify_embeddings; verify_embeddings('training/data/processed/embedded/', 5)"
   python -c "from training.scripts.preprocessing.stage6_graphs import verify_h5; verify_h5('training/data/graphs/all_graphs.h5', 5)"
   ```

3. **Next story**: Stage 7 (CFA-Aware Split) then Story 8 (Model Training)

In [ ]:
# Final disk usage summary
!echo "=== Disk Usage ==="
!du -sh {CPG_DIR} {EMBED_DIR} {GRAPH_DIR} 2>/dev/null
!echo ""
!echo "=== GPU Memory Peak ==="

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak GPU memory: {peak:.2f} GB")